We import all required modules

In [ ]:
import os
import shutil
import tarfile
import glob
import yaml
from sklearn.model_selection import train_test_split

from gcnn.graph_database import build_graph_database
from gcnn.graphs import set_up_molecular_graphs
from gcnn.features import set_up_features
from gcnn.paths import (
    DATA_DIR, ORIGINAL_DATASET, DATASET,
    TRAINING_SET, VALIDATION_SET, TEST_SET, PROCESSED_DIR,
    CONFIG_DIR, ensure_data_dirs,
)

We define the required paths

In [2]:
# All paths are resolved automatically by gcnn.paths
# Override with GCNN_DATA_DIR env var if needed

ensure_data_dirs()

print(f"Data directory: {DATA_DIR}")

Data directory: /Users/migueldc/Documents/git_projects/gcnn/data


We download the database in our directory

In [ ]:
tar_path = ORIGINAL_DATASET / "dsgdb9nsd.xyz.tar"

# Only extract if not already done
if not any(DATASET.glob("dsgdb9nsd_*.xyz")):
    with tarfile.open(tar_path) as my_tar:
        my_tar.extractall(DATASET, filter="data")
    print("Extraction complete.")
else:
    print("Dataset already extracted, skipping.")

files = glob.glob(str(DATASET / "dsgdb9nsd_*.xyz"))

print(f"Total number of entries: {len(files)}")

For this proof-of-principle calculation we are going to work only with 5% of the original dataset (i.e., instead of the original 140K, we will work with about 7K samples)

In [4]:
# # Optional: use a subset for quick experiments
# smaller_dataset, _ = train_test_split(files, test_size=0.95, random_state=42)
# files = smaller_dataset
# print(f"Using smaller dataset: {len(files)} entries")


We now split the smaller database into train (80%), validate(10%) and test (10%) sets, and store them in directories

In [ ]:
reminder_set, test = train_test_split(files, test_size=0.1, random_state=42)
train, validate = train_test_split(reminder_set, test_size=0.1, random_state=42)

print(f"test_size = {len(test)}")
print(f"validate_size = {len(validate)}")
print(f"train_size = {len(train)}")

total = len(test) + len(validate) + len(train)
print(f"total_size = {total}")
assert total == len(files), "Split sizes do not add up!"

Now, we just move the right files to the corresponding directories for the smaller size proof-of-concept

In [6]:
# Only move if destination directories are empty
if not any(TEST_SET.glob("*.xyz")):
    for file in test:
        shutil.move(file, str(TEST_SET))
    for file in validate:
        shutil.move(file, str(VALIDATION_SET))
    for file in train:
        shutil.move(file, str(TRAINING_SET))
    print("Files moved to train/validation/test directories.")
else:
    print("Splits already exist, skipping move.")


Splits already exist, skipping move.


### Pre-compute graph database

We now build the graph representation of each molecule **once** and save it as a single `.pt` file per split.
This eliminates ~99% of redundant computation during training (graphs are static across epochs).

In [7]:
# Load experiment config
with open(CONFIG_DIR / "sample.yml") as f:
    cfg = yaml.safe_load(f)

# Build the graph constructor from config
edges, bond_angle, dihedral = set_up_features(cfg)
graphs = set_up_molecular_graphs(
    cfg.get("graphType", "covalent"),
    edge_features=edges,
    bond_angle_features=bond_angle,
    dihedral_features=dihedral,
    node_feature_list=cfg["nodeFeatures"],
    n_total_node_features=cfg["nTotalNodeFeatures"],
    n_max_neighbours=cfg.get("nMaxNeighbours", 6),
)


In [8]:
# Build train/val/test graph databases (only if they don't already exist)
splits = [
    (TRAINING_SET, PROCESSED_DIR / "train_graphs.pt", cfg.get("nTrainMaxEntries")),
    (VALIDATION_SET, PROCESSED_DIR / "val_graphs.pt", cfg.get("nValMaxEntries")),
    (TEST_SET, PROCESSED_DIR / "test_graphs.pt", cfg.get("nValMaxEntries")),
]

for xyz_dir, pt_path, n_max in splits:
    if pt_path.exists():
        print(f"Skipping {pt_path.name} (already exists)")
        continue
    print(f"Building {pt_path.name} from {xyz_dir.name}...")
    build_graph_database(
        xyz_dir=xyz_dir,
        output_path=pt_path,
        graphs=graphs,
        n_max_entries=n_max,
        seed=cfg.get("randomSeed", 42),
        config=cfg,
    )


Skipping train_graphs.pt (already exists)
Skipping val_graphs.pt (already exists)
Skipping test_graphs.pt (already exists)
